In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/covid_19_clean_complete.csv")

In [3]:
province = df["Province/State"].to_numpy()
print(province)

[nan nan nan ... nan nan nan]


In [4]:
country = df["Country/Region"].to_numpy()
print(country)

['Afghanistan' 'Albania' 'Algeria' ... 'Comoros' 'Tajikistan' 'Lesotho']


In [5]:
lat = df["Lat"].to_numpy()
print(lat)

[ 33.93911  41.1533   28.0339  ... -11.6455   38.861   -29.61   ]


In [7]:
long = df["Long"].to_numpy()
print(long)

[67.709953 20.1683    1.6596   ... 43.3333   71.2761   28.2336  ]


In [8]:
date = df["Date"].to_numpy()
print(date)

['2020-01-22' '2020-01-22' '2020-01-22' ... '2020-07-27' '2020-07-27'
 '2020-07-27']


In [6]:
confrimed = df["Confirmed"].to_numpy()
print(confrimed)

[   0    0    0 ...  354 7235  505]


In [9]:
deaths = df["Deaths"].to_numpy()
print(deaths)

[ 0  0  0 ...  7 60 12]


In [10]:
recovered = df["Recovered"].to_numpy()
print(recovered)

[   0    0    0 ...  328 6028  128]


In [12]:
active = df["Active"].to_numpy()
print(active)

[   0    0    0 ...   19 1147  365]


In [11]:
who = df["WHO Region"].to_numpy()
print(who)

['Eastern Mediterranean' 'Europe' 'Africa' ... 'Africa' 'Europe' 'Africa']


In [ ]:
print('Missing values before cleaning:')
display(df.isnull().sum())

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
print('Data type of Date column after conversion:')
display(df['Date'].dtype)

In [ ]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
final_rows = df.shape[0]
print(f"Number of duplicate records removed: {initial_rows - final_rows}")

In [ ]:
print('Data types of all columns:')
display(df.info())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate confirmed cases by date
daily_cases = df.groupby('Date')['Confirmed'].sum().reset_index()

# Calculate daily new cases
daily_cases['New_Confirmed'] = daily_cases['Confirmed'].diff().fillna(0)

# Plot Confirmed Cases over Time (Cumulative)
plt.figure(figsize=(15, 7))
sns.lineplot(x='Date', y='Confirmed', data=daily_cases)
plt.title('Cumulative Confirmed COVID-19 Cases Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Confirmed Cases')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Daily New Confirmed Cases
plt.figure(figsize=(15, 7))
sns.lineplot(x='Date', y='New_Confirmed', data=daily_cases)
plt.title('Daily New Confirmed COVID-19 Cases Over Time')
plt.xlabel('Date')
plt.ylabel('Daily New Confirmed Cases')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Aggregate data by country
country_summary = df.groupby('Country/Region')[['Confirmed', 'Deaths', 'Recovered', 'Active']].sum().reset_index()

# Sort by Confirmed cases to identify most affected countries
country_summary_sorted_confirmed = country_summary.sort_values(by='Confirmed', ascending=False).head(10)

# Plot Top 10 Countries by Confirmed Cases
plt.figure(figsize=(15, 7))
sns.barplot(x='Country/Region', y='Confirmed', data=country_summary_sorted_confirmed, palette='viridis')
plt.title('Top 10 Countries by Confirmed COVID-19 Cases')
plt.xlabel('Country/Region')
plt.ylabel('Total Confirmed Cases')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sort by Deaths to identify most affected countries by deaths
country_summary_sorted_deaths = country_summary.sort_values(by='Deaths', ascending=False).head(10)

# Plot Top 10 Countries by Deaths
plt.figure(figsize=(15, 7))
sns.barplot(x='Country/Region', y='Deaths', data=country_summary_sorted_deaths, palette='mako')
plt.title('Top 10 Countries by COVID-19 Deaths')
plt.xlabel('Country/Region')
plt.ylabel('Total Deaths')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Sort by Recovered to identify most affected countries by recoveries
country_summary_sorted_recovered = country_summary.sort_values(by='Recovered', ascending=False).head(10)

# Plot Top 10 Countries by Recovered Cases
plt.figure(figsize=(15, 7))
sns.barplot(x='Country/Region', y='Recovered', data=country_summary_sorted_recovered, palette='rocket')
plt.title('Top 10 Countries by COVID-19 Recovered Cases')
plt.xlabel('Country/Region')
plt.ylabel('Total Recovered Cases')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
!pip install dash jupyter_dash plotly

# Restart runtime after installation if running for the first time in a session
print("Please restart the runtime (Runtime > Restart runtime) if this is your first time installing these libraries in this session.")

In [ ]:
from jupyter_dash import JupyterDash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import plotly.express as px

# Initialize the Dash app
app = JupyterDash(__name__)

In [ ]:
# Ensure 'Date' is datetime for date range picker
df['Date'] = pd.to_datetime(df['Date'])

# Get unique countries for the dropdown
country_options = [{'label': country, 'value': country} for country in sorted(df['Country/Region'].unique())]

app.layout = html.Div([
    html.H1("COVID-19 Dashboard", style={'textAlign': 'center'}),

    html.Div([
        html.H3("Select Country:", style={'marginRight': '10px'}),
        dcc.Dropdown(
            id='country-dropdown',
            options=country_options,
            value='US',  # Default value
            multi=False,
            style={'width': '50%'}
        )
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '20px'}),

    html.Div([
        html.H3("Select Date Range:", style={'marginRight': '10px'}),
        dcc.DatePickerRange(
            id='date-range-picker',
            start_date=df['Date'].min(),
            end_date=df['Date'].max(),
            display_format='YYYY-MM-DD'
        )
    ], style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'marginBottom': '20px'}),

    dcc.Graph(id='live-update-graph')
])

In [ ]:
@app.callback(
    Output('live-update-graph', 'figure'),
    [Input('country-dropdown', 'value'),
     Input('date-range-picker', 'start_date'),
     Input('date-range-picker', 'end_date')]
)
def update_graph_live(selected_country, start_date, end_date):
    filtered_df = df[df['Country/Region'] == selected_country]
    filtered_df = filtered_df[
        (filtered_df['Date'] >= start_date) &
        (filtered_df['Date'] <= end_date)
    ]

    # Aggregate confirmed cases by date for the selected country and date range
    daily_cases_country = filtered_df.groupby('Date')['Confirmed'].sum().reset_index()

    fig = px.line(
        daily_cases_country,
        x='Date',
        y='Confirmed',
        title=f'Cumulative Confirmed COVID-19 Cases in {selected_country}',
        labels={'Confirmed': 'Cumulative Confirmed Cases', 'Date': 'Date'}
    )
    fig.update_layout(xaxis_title='Date', yaxis_title='Cumulative Confirmed Cases')
    return fig

In [ ]:
# Run the app in development mode
if __name__ == '__main__':
    app.run_server(mode='inline', port=8050, debug=True)

In [ ]:
fig_conf_deaths = px.scatter(df, x='Confirmed', y='Deaths',
                             title='Confirmed Cases vs. Deaths',
                             labels={'Confirmed': 'Total Confirmed Cases', 'Deaths': 'Total Deaths'})
fig_conf_deaths.show()

# Calculate and print correlation
correlation_conf_deaths = df['Confirmed'].corr(df['Deaths'])
print(f"Correlation between Confirmed Cases and Deaths: {correlation_conf_deaths:.2f}")

In [ ]:
fig_conf_recovered = px.scatter(df, x='Confirmed', y='Recovered',
                              title='Confirmed Cases vs. Recovered Cases',
                              labels={'Confirmed': 'Total Confirmed Cases', 'Recovered': 'Total Recovered Cases'})
fig_conf_recovered.show()

# Calculate and print correlation
correlation_conf_recovered = df['Confirmed'].corr(df['Recovered'])
print(f"Correlation between Confirmed Cases and Recovered Cases: {correlation_conf_recovered:.2f}")

In [ ]:
fig_active_conf = px.scatter(df, x='Active', y='Confirmed',
                           title='Active Cases vs. Confirmed Cases',
                           labels={'Active': 'Total Active Cases', 'Confirmed': 'Total Confirmed Cases'})
fig_active_conf.show()

# Calculate and print correlation
correlation_active_conf = df['Active'].corr(df['Confirmed'])
print(f"Correlation between Active Cases and Confirmed Cases: {correlation_active_conf:.2f}")

In [ ]:
# Aggregate total Confirmed, Deaths, and Recovered cases globally
total_cases = df[['Confirmed', 'Deaths', 'Recovered']].sum().reset_index()
total_cases.columns = ['Case_Type', 'Count']

fig_pie_distribution = px.pie(total_cases, values='Count', names='Case_Type',
                              title='Global COVID-19 Case Distribution',
                              hole=0.3)
fig_pie_distribution.show()

In [ ]:
numerical_cols = ['Confirmed', 'Deaths', 'Recovered', 'Active', 'Lat', 'Long']
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numerical Features')
plt.show()